In [ ]:
# input
posi_domain = "../../tmp/entryId-tedIds-posis.tsv"
high_conf_pred = "../../../predict_afdb/data/pred_ge_3_clique_3.tsv"
# output
site_domain = "./tmp/entryId-sites-tedIds.tsv"

In [2]:
import pandas as pd
df = pd.read_table(posi_domain, header=None, names=["seq_id", "ted_ids", "posis"])

In [3]:
from tqdm import tqdm
posi2domain = dict()
for _, row in tqdm(df.iterrows(), total=len(df)):
    seq_id = row['seq_id']
    domains = row['ted_ids'].split(",")
    posis = row['posis'].split(";")
    for i in range(len(domains)):
        domain = domains[i]
        for posi in posis[i].split(","):
            posi2domain[(seq_id, posi)] = domain

100%|██████████| 35343707/35343707 [28:47<00:00, 20458.43it/s] 


In [5]:
from itertools import combinations
import networkx as nx

def get_site(site_str: str):
    g = nx.Graph()
    sites = site_str.split(";")
    for s in sites:
        members = s.split(",")
        for (i, j) in combinations(members, 2):
            g.add_edge(i, j)
    
    result: list[list[str]] = []
    for s in nx.connected_components(g):
        result.append(sorted(list(s)))
    return result

import csv
with open(site_domain, "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t", lineterminator="\n")
    df = pd.read_table(high_conf_pred, usecols=["seq_id", "site"])

    for _, row in tqdm(df.iterrows(), total=len(df)):
        seq_id = row["seq_id"].split("-")[1]
        sites = get_site(row["site"])
        sites_domains: list[list[str]] = []
        for site in sites:
            s_domains = set()
            for posi in site:
                key = (seq_id, posi)
                domain = posi2domain[key] if key in posi2domain else "None"
                s_domains.add(domain)
            sites_domains.append(list(s_domains))
        
        sites_str = ";".join([",".join(i) for i in sites])
        sites_domains_str = ";".join([",".join(i) for i in sites_domains])
        _ = writer.writerow([seq_id, sites_str, sites_domains_str])

100%|██████████| 38361041/38361041 [48:50<00:00, 13091.73it/s]
